# Phase 7 - SMOKE COMPARE - 3-config Stage 2 retrain trajectory probe

**Goal**: Pick the Stage 2 retrain config with the best trajectory to beat
the non-causal baseline (Pearson > 0.815, RMSE < ref, F1@p99 > 0.55) at
full-scale training (~200 epochs on A100).

**Strategy**: train 3 candidate configs for 3 epochs each on top of the
FROZEN Phase 6 dual-path Stage 1 (encoder + RCN + regression_head +
dual_path), compute a fast BS30-light Pearson per epoch, then extrapolate.

| Config | Warm-start | LR     | Warmup | Notes                                  |
|--------|------------|--------|--------|----------------------------------------|
| A      | False      | 2e-4   | 0      | Mirror non-causal exact hyperparams    |
| B      | True       | 5e-5   | 0      | Prudent fine-tune from existing S2     |
| C      | False      | 3e-4   | 500    | Aggressive from-scratch + warmup       |

**Reuses**:
- `train_epoch_stage2_cached` (proven path, 28h non-causal run used same fn)
- `precompute_stage1_outputs_variant` pattern adapted for dual-path mu_total
- Phase 6 sigma_data = 0.193 (recalibrated for dual-path Stage 1 regime)

**Budget**:
- First run (cache cold): ~75-90 min on A100
- Subsequent runs (cache warm): ~45 min on A100
- mu_total cache: ~1.4 GB on Drive, computed once


In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab - Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')


In [ ]:
# === Cell 2 : Constants + 3-config grid ===
import json
import numpy as np
import torch
from pathlib import Path
from omegaconf import OmegaConf

# ----- Drive paths -----
DRIVE_ROOT       = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N        = DRIVE_ROOT / 'oracle_9node' / 'seed_42'

# Stage 1 dual-path checkpoint (Phase 6 frozen).
CKPT_DUALPATH    = ORACLE_9N / 'epoch_best_dualpath.pth'

# Warm-start source for Config B (Stage 2 from oracle_9node).
CKPT_STAGE2_WARMSTART = ORACLE_9N / 'epoch_last.pth'

# Cached Stage 1 outputs for Stage 2 (mu_total via dual_path).
STAGE1_CACHE_PATH = ORACLE_9N / 'stage1_cache_phase7_dualpath.pt'

# Output dir for smoke comparison artefacts.
SMOKE_OUT_DIR = ORACLE_9N / 'phase7_smoke_compare'
SMOKE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Phase 6 dualpath-recalibrated sigma_data.
SIGMA_DATA_NEW = 0.193

# ----- 3-config grid -----
CONFIGS = [
    {
        'name': 'A_scratch_exact',
        'description': 'From-scratch + non-causal exact hyperparams',
        'warm_start': False,
        'lr': 2e-4,
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'warmup_steps': 0,
        'amp_mode': 'cuda_bf16',
        'batch_size': 64,
        'gradient_checkpointing': True,
    },
    {
        'name': 'B_warmstart_prudent',
        'description': 'Warm-start Stage 2 + low LR (fine-tune for distribution shift)',
        'warm_start': True,
        'lr': 5e-5,
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'warmup_steps': 0,
        'amp_mode': 'cuda_bf16',
        'batch_size': 64,
        'gradient_checkpointing': True,
    },
    {
        'name': 'C_scratch_aggressive',
        'description': 'From-scratch + higher LR + warmup',
        'warm_start': False,
        'lr': 3e-4,
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'warmup_steps': 500,
        'amp_mode': 'cuda_bf16',
        'batch_size': 64,
        'gradient_checkpointing': True,
    },
]
SMOKE_EPOCHS_PER_CONFIG = 3

# ----- Light-eval hyperparams (smoke-tuned -- not BS30 full 16/64/18) -----
# Cell 13 (Test D) measured 17.6 min for the full setting -- intractable
# for a smoke. We compare TRAJECTORIES, not absolute Pearson, so a much
# cheaper eval is fine. Total cost : ~20s per epoch (vs ~17.6 min).
LIGHT_EVAL_N_BATCHES = 2     # was 4
LIGHT_EVAL_K_SAMPLES = 4     # was 16
LIGHT_EVAL_N_STEPS   = 8     # was 18 -- still well above Heun convergence floor

# ----- Non-causal target metrics (baseline to beat) -----
NONCAUSAL_TARGET_PEARSON = 0.815

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[Cell 2] DEVICE              = {DEVICE}')
print(f'[Cell 2] CKPT_DUALPATH       = {CKPT_DUALPATH}  exists={CKPT_DUALPATH.exists()}')
print(f'[Cell 2] CKPT_STAGE2_WARMSTART={CKPT_STAGE2_WARMSTART}  exists={CKPT_STAGE2_WARMSTART.exists()}')
print(f'[Cell 2] STAGE1_CACHE_PATH   = {STAGE1_CACHE_PATH}  exists={STAGE1_CACHE_PATH.exists()}')
print(f'[Cell 2] SMOKE_OUT_DIR       = {SMOKE_OUT_DIR}')
print(f'[Cell 2] CONFIGS             = {[c["name"] for c in CONFIGS]}')
print(f'[Cell 2] SMOKE_EPOCHS_PER_CONFIG = {SMOKE_EPOCHS_PER_CONFIG}')
print(f'[Cell 2] LIGHT_EVAL : N_BATCHES={LIGHT_EVAL_N_BATCHES} K={LIGHT_EVAL_K_SAMPLES} STEPS={LIGHT_EVAL_N_STEPS}')


In [ ]:
# === Cell 3 : Config + Pipeline + Dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config : merge base + corrdiff_normal (== non-causal stage 2 hyperparams) ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# Stage 1 dataloader stays at BS=1 (we iterate samples once to fill the cache).
# Stage 2 will use a SEPARATE cached dataloader at BS=64 (see Cell 5).
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN             = int(CONFIG.data.seq_len)
BASELINE_STRATEGY   = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR     = int(CONFIG.data.baseline_factor)
NORMALIZE           = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY   = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = 1
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
            elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
            elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
            else:              dynamic_features[nt] = _ensure_2d(lr0)
    else:
        dynamic_features = {nt: _ensure_2d(lr0) for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])
print(f'[Cell 3] HR shape = ({H_HR}, {W_HR})')


In [ ]:
# === Cell 4 : Load Stage 1 dual-path (encoder + RCN + head + dual_path) FROZEN ===
from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.training.stage1_paths import batch_lr_grid_last
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder


def _parse_encoder_metapaths_from_ckpt(enc_sd):
    seen, order = {}, []
    for k in enc_sd:
        if not k.startswith('metapath_convs.'):
            continue
        rest = k[len('metapath_convs.'):]
        parts = rest.split('__')
        if len(parts) < 4:
            continue
        name = parts[0]; src = parts[1]; rel = parts[2]; tgt = parts[3].split('.')[0]
        if name not in seen:
            seen[name] = (src, rel, tgt); order.append(name)
    return [(n,) + seen[n] for n in order]


def _clean_sd(sd):
    if sd is None:
        return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd


def _safe_load(module, ck_data, keys, label):
    for key in keys:
        sd = ck_data.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                missing, unexpected = module.load_state_dict(sd, strict=False)
                msg = f'  [{label}] loaded from "{key}"'
                if missing:    msg += f'  | missing={len(missing)}'
                if unexpected: msg += f'  | unexpected={len(unexpected)}'
                print(msg)
                return True
            except Exception as e:
                print(f'  [{label}] FAILED with "{key}" : {type(e).__name__}: {e}')
                continue
    print(f'  [{label}] no valid key found in {keys}')
    return False


def _strip_prefixes(sd):
    if sd is None:
        return None
    prefixes = ['_orig_mod.', 'module.']
    out = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out


print(f'[Cell 4] Loading Stage 1 dual-path : {CKPT_DUALPATH}')
ck_s1 = torch.load(CKPT_DUALPATH, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] ckpt keys[:12] = {sorted(ck_s1.keys())[:12]}')

enc_sd = _clean_sd(ck_s1.get('encoder_state_dict', {}))
parsed = _parse_encoder_metapaths_from_ckpt(enc_sd)
print(f'  metapaths detected : {[t[0] for t in parsed]}')
cfgs = [
    IntelligibleVariableConfig(name=n, meta_path=(s, r, t), pool='mean')
    for n, s, r, t in parsed
]
encoder = IntelligibleVariableEncoder(
    configs=cfgs,
    hidden_dim=int(CONFIG.encoder.hidden_dim),
    conditioning_dim=int(CONFIG.encoder.conditioning_dim),
).to(DEVICE)
n_vars = len(cfgs)
num_vars = n_vars

_probe_b = next(iter(val_dataset))
_lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
rcn_driver_dim = _lr_nodes.shape[-1]

rcn_cell = RCNCell(
    num_vars=n_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=rcn_driver_dim,
    reconstruction_dim=rcn_driver_dim,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

PATH_B_KIND          = 'unet'
PATH_B_UNET_CHANNELS = (32, 64, 128)
PATH_B_UNET_LR_SHAPE = (23, 26)
PATH_B_BASE_CH       = 48
GATE_MAX_MEAN        = 0.40
dual_path = DualPathPredictor(
    in_channels=C_LR,
    base_ch=PATH_B_BASE_CH,
    hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=GATE_MAX_MEAN,
    path_b_kind=PATH_B_KIND,
    path_b_unet_channels=PATH_B_UNET_CHANNELS,
    path_b_unet_lr_shape=PATH_B_UNET_LR_SHAPE,
).to(DEVICE)

_safe_load(encoder,         ck_s1, ['encoder_state_dict'],                            'encoder')
_safe_load(rcn_cell,        ck_s1, ['rcn_cell_state_dict', 'rcn_state_dict'],         'rcn_cell')
_safe_load(regression_head, ck_s1, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
_safe_load(dual_path,       ck_s1, ['dual_path_state_dict'],                          'dual_path')

# FREEZE all Stage 1 modules.
_stage1_n_total = 0
_stage1_n_trainable = 0
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters():
        p.requires_grad_(False)
        _stage1_n_total += p.numel()
    m.eval()
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters():
        if p.requires_grad:
            _stage1_n_trainable += p.numel()
assert _stage1_n_trainable == 0, (
    f'Stage 1 freeze failed : {_stage1_n_trainable} trainable params remain'
)
print(f'  [verify] Stage 1 frozen : {_stage1_n_total:,} params, '
      f'{_stage1_n_trainable} trainable (must be 0).')

_rcn_core = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
if hasattr(_rcn_core, 'A_dag'):
    _A_dag = _rcn_core.A_dag.detach()
    print(f'  A_dag shape={tuple(_A_dag.shape)}  norm={_A_dag.norm():.4f}  '
          f'asym={(_A_dag - _A_dag.T).abs().mean():.4f}')

print(f'[Cell 4] Stage 1 dual-path FROZEN. num_vars = {num_vars}')


# Helper used by precompute loop to compute mu_total via dual_path.
@torch.no_grad()
def predict_mu_total(_batch):
    '''Stage 1 forward (frozen): returns (mu_total, baseline_log, hr_residual).

    mu_total = mu_A + gate * mu_B(LR)   [B, 1, H, W]
    baseline_log = baseline at last step [B, 1, H, W]
    hr_residual  = HR_log1p - baseline   [B, 1, H, W]  (= sample['residual'][-1])
    '''
    lr_data = _batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(_batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    mu_A    = regression_head(seq_out.states[-1])
    if mu_A.dim() == 3:
        mu_A = mu_A.unsqueeze(0)
    lr_grid = batch_lr_grid_last(_batch, builder=builder, device=DEVICE)
    lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
    mu_total, _mu_B, _gate = dual_path(lr_safe, mu_A)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)

    bl = _batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)

    hr_res = _batch['residual'][-1].to(DEVICE)
    if hr_res.dim() == 3:
        hr_res = hr_res.unsqueeze(0)
    return mu_total, bl, hr_res


In [ ]:
# === Cell 5 : Precompute mu_total cache + build cached dataloaders ===
# This cell runs the FROZEN Stage 1 dual-path once over the train_dataset,
# stores (mu_HR=mu_total, baseline_log, delta_target=residual - mu_total,
# valid_mask) on Drive, then exposes a map-style _CachedDataset + DataLoader
# at batch_size=64 -- reused by all 3 smoke configs.
import time

# Copy cache from Drive to local SSD (avoid mmap/page-fault stalls on Drive)
import shutil as _shutil_phase7
LOCAL_CACHE_PATH = Path('/content/stage1_cache_phase7_dualpath.pt')
if STAGE1_CACHE_PATH.exists() and (not LOCAL_CACHE_PATH.exists() or LOCAL_CACHE_PATH.stat().st_size != STAGE1_CACHE_PATH.stat().st_size):
    print(f'[Cell 5] Copying cache Drive -> local SSD (avoid mmap stall)...')
    import time as _t
    _t0 = _t.time()
    _shutil_phase7.copy(str(STAGE1_CACHE_PATH), str(LOCAL_CACHE_PATH))
    print(f'  Done in {_t.time()-_t0:.1f}s ({LOCAL_CACHE_PATH.stat().st_size/1e9:.2f} GB)')

if LOCAL_CACHE_PATH.exists():
    print(f'[Cell 5] Loading cache from local SSD {LOCAL_CACHE_PATH}')
    cache = torch.load(LOCAL_CACHE_PATH, map_location='cpu', weights_only=False)
    # Force materialize tensors into dense RAM (break any mmap/lazy backing)
    cache = {k: v.contiguous().clone() for k, v in cache.items()}
    _gb = sum(v.element_size()*v.nelement() for v in cache.values())/1e9
    print(f'[Cell 5] Cache rehydrated to dense RAM ({_gb:.2f} GB)')
elif STAGE1_CACHE_PATH.exists():
    print(f'[Cell 5] Loading existing cache {STAGE1_CACHE_PATH}')
    cache = torch.load(STAGE1_CACHE_PATH, map_location='cpu', weights_only=False)
    cache = {k: v.contiguous().clone() for k, v in cache.items()}
else:
    print('[Cell 5] Precomputing mu_total cache (one-time, ~30-45 min on A100)...')
    mu_HR_list, baseline_log_list, delta_target_list, valid_mask_list = [], [], [], []
    _t0 = time.time()

    _n_iter = len(train_dataset) if hasattr(train_dataset, '__len__') else None
    for i, sample in enumerate(train_dataset):
        batch = convert_sample_to_batch(sample, builder, DEVICE)
        with torch.no_grad():
            mu_total, bl, hr_res = predict_mu_total(batch)
        # delta_target = (HR - baseline) - mu_total = residual - mu_total
        delta_target = hr_res - mu_total
        valid_mask = torch.isfinite(hr_res)

        # Strip batch dim (1) since dataset items are per-sample.
        mu_HR_list.append(mu_total.squeeze(0).cpu())
        baseline_log_list.append(bl.squeeze(0).cpu())
        delta_target_list.append(torch.nan_to_num(delta_target, nan=0.0).squeeze(0).cpu())
        valid_mask_list.append(valid_mask.squeeze(0).cpu())

        if (i + 1) % 500 == 0:
            _elapsed = time.time() - _t0
            _rate = (i + 1) / max(_elapsed, 1e-6)
            print(f'  {i + 1}{f"/{_n_iter}" if _n_iter else ""} cached '
                  f'| {_elapsed / 60:.1f} min | {_rate:.1f} samp/s')

    cache = {
        'mu_HR':        torch.stack(mu_HR_list, dim=0),
        'baseline_log': torch.stack(baseline_log_list, dim=0),
        'delta_target': torch.stack(delta_target_list, dim=0),
        'valid_mask':   torch.stack(valid_mask_list, dim=0),
    }
    STAGE1_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(cache, STAGE1_CACHE_PATH)
    print(f'[Cell 5] Cache saved : {STAGE1_CACHE_PATH}')

print(f'[Cell 5] Cache shapes :')
print(f'  mu_HR        = {tuple(cache["mu_HR"].shape)}')
print(f'  baseline_log = {tuple(cache["baseline_log"].shape)}')
print(f'  delta_target = {tuple(cache["delta_target"].shape)}')
print(f'  valid_mask   = {tuple(cache["valid_mask"].shape)}')
print(f'[Cell 5] delta_target stats : mean={cache["delta_target"].mean():.4f} '
      f'std={cache["delta_target"].std():.4f}')


# Map-style dataset (yields dicts compatible with train_epoch_stage2_cached).
class _CachedDataset(torch.utils.data.Dataset):
    def __init__(self, cache, indices=None):
        self.mu_HR        = cache['mu_HR']
        self.baseline_log = cache['baseline_log']
        self.delta_target = cache['delta_target']
        self.valid_mask   = cache['valid_mask']
        self.indices = indices if indices is not None else list(range(len(self.mu_HR)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return {
            'mu_HR':        self.mu_HR[idx],
            'baseline_log': self.baseline_log[idx],
            'delta_target': self.delta_target[idx],
            'valid_mask':   self.valid_mask[idx],
        }


N_total = int(cache['mu_HR'].shape[0])
VAL_FRACTION = 0.1
N_train = int(N_total * (1.0 - VAL_FRACTION))
train_indices = list(range(N_train))
val_indices   = list(range(N_train, N_total))

train_cached_dataset = _CachedDataset(cache, train_indices)
val_cached_dataset   = _CachedDataset(cache, val_indices)

STAGE2_BATCH_SIZE = 64
train_cached_dataloader = torch.utils.data.DataLoader(
    train_cached_dataset,
    batch_size=STAGE2_BATCH_SIZE,
    shuffle=True,
    num_workers=0,         # was 4 - workers cause stall on mmap'd cache
    pin_memory=False,      # was True - pin on mmap triggers memcpy each access
    drop_last=False,
)
val_cached_dataloader = torch.utils.data.DataLoader(
    val_cached_dataset,
    batch_size=STAGE2_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)
print(f'[Cell 5] Splits  : train={len(train_cached_dataset)}  val={len(val_cached_dataset)}')
print(f'[Cell 5] Loaders : train_bs={STAGE2_BATCH_SIZE}  val_bs={STAGE2_BATCH_SIZE}')


In [ ]:
# === Cell 6 : build_fresh_stage2 helper + light_eval helper ===
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

# Probe HR channels (1 for precip).
_probe_sample = next(iter(val_dataset))
hr_channels = int(_probe_sample['residual'].shape[1])
print(f'[Cell 6] hr_channels = {hr_channels}')


def build_fresh_stage2(warm_start: bool, gradient_checkpointing: bool):
    '''Build a CausalDiffusionDecoder for one smoke config.

    EDM config inherits tail_weight + S_churn from CONFIG.diffusion.edm
    (== non-causal corrdiff_normal preset). sigma_data is then overridden
    to the Phase 6 dual-path recalibrated value (0.193).
    '''
    edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
    UNET_KWARGS = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
    for _k in ('down_block_types', 'up_block_types'):
        if _k in UNET_KWARGS and isinstance(UNET_KWARGS[_k], list):
            UNET_KWARGS[_k] = tuple(UNET_KWARGS[_k])
    UNET_KWARGS['projection_class_embeddings_input_dim'] = (
        num_vars * int(CONFIG.diffusion.conditioning_dim))

    diff = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height),
        width=int(CONFIG.diffusion.width),
        unet_kwargs=UNET_KWARGS,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=bool(gradient_checkpointing),
        conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
        anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
        edm_config=edm_cfg,
        causal_concat=True,
    ).to(DEVICE)

    if warm_start:
        if not CKPT_STAGE2_WARMSTART.exists():
            raise FileNotFoundError(
                f'Warm-start ckpt missing : {CKPT_STAGE2_WARMSTART}'
            )
        ck = torch.load(CKPT_STAGE2_WARMSTART, map_location=DEVICE, weights_only=False)
        diff_sd = _strip_prefixes(ck.get('diffusion_state_dict'))
        if diff_sd is None:
            raise RuntimeError(
                f'diffusion_state_dict absent from {CKPT_STAGE2_WARMSTART}'
            )
        info = diff.load_state_dict(diff_sd, strict=False)
        print(f'  [build_stage2] warm-start loaded  '
              f'missing={len(info.missing_keys)}  unexpected={len(info.unexpected_keys)}')
        del ck, diff_sd
    else:
        print('  [build_stage2] from scratch (random init)')

    # Override sigma_data with Phase 6 dualpath recalibrated value.
    diff.edm_config.sigma_data = float(SIGMA_DATA_NEW)
    print(f'  [build_stage2] sigma_data set to {diff.edm_config.sigma_data}')

    for p in diff.parameters():
        p.requires_grad_(True)
    diff.train()
    _n = sum(p.numel() for p in diff.parameters())
    print(f'  [build_stage2] diffusion params : {_n:,}')
    return diff


@torch.no_grad()
def light_eval(diff_decoder, val_loader):
    '''BS30-LIGHT: 4 batches x 16 K-samples x 18 steps using the cached val set.

    Computes Pearson + RMSE on the full HR reconstruction
        pred_full   = baseline_log + mu_HR + delta_pred
        target_full = baseline_log + mu_HR + delta_target
    in log1p space (same units as Stage 2 training).
    '''
    diff_decoder.eval()
    preds   = []
    targets = []
    n_done  = 0

    for batch in val_loader:
        if n_done >= LIGHT_EVAL_N_BATCHES:
            break
        mu_HR        = batch['mu_HR'].to(DEVICE)
        baseline_log = batch['baseline_log'].to(DEVICE)
        delta_target = batch['delta_target'].to(DEVICE)

        samples = []
        # bf16 autocast in sample() : measured ~3x speedup on A100
        # vs fp32 default (negligible accuracy impact for smoke ranking).
        for _k in range(LIGHT_EVAL_K_SAMPLES):
            with torch.autocast('cuda', dtype=torch.bfloat16):
                out = diff_decoder.sample(
                    conditioning=None,
                    num_steps=LIGHT_EVAL_N_STEPS,
                    scheduler_type='edm_karras',
                    apply_constraints=False,
                    mu_HR=mu_HR,
                    baseline_log=baseline_log,
                )
            res = out.residual if hasattr(out, 'residual') else out
            samples.append(res)
        ens = torch.stack(samples, dim=0)
        delta_pred = ens.mean(dim=0)

        pred_full   = baseline_log + mu_HR + delta_pred
        target_full = baseline_log + mu_HR + delta_target

        preds.append(pred_full.cpu())
        targets.append(target_full.cpu())
        n_done += 1

    diff_decoder.train()

    p = torch.cat(preds,   dim=0).flatten()
    t = torch.cat(targets, dim=0).flatten()
    valid = torch.isfinite(t) & torch.isfinite(p)
    if valid.sum() < 2:
        return {'pearson': float('nan'), 'rmse': float('nan')}
    pv = p[valid]; tv = t[valid]
    pm, tm = pv.mean(), tv.mean()
    num = ((pv - pm) * (tv - tm)).sum()
    den = (((pv - pm) ** 2).sum() * ((tv - tm) ** 2).sum()).sqrt()
    pearson = float(num / max(den, torch.tensor(1e-12)))
    rmse    = float((pv - tv).pow(2).mean().sqrt())
    return {'pearson': pearson, 'rmse': rmse}


print('[Cell 6] build_fresh_stage2 + light_eval ready')


In [ ]:
# === Cell 7 : Main smoke loop (3 configs x 3 epochs) ===
# CRASH-PROOF :
#   - dumps SMOKE_OUT_DIR/smoke_partial.json after EACH epoch
#   - on re-run, reloads partial + SKIPS configs already fully done
from st_cdgm.training.two_stage import train_epoch_stage2_cached
import time, gc, json as _json

PARTIAL_PATH = SMOKE_OUT_DIR / 'smoke_partial.json'

# ---- Resume : load partial if any ----
if PARTIAL_PATH.exists():
    try:
        results = _json.loads(PARTIAL_PATH.read_text())
        print(f'[Cell 7] RESUMED from {PARTIAL_PATH}  -> {list(results.keys())}')
    except Exception as _e:
        print(f'[Cell 7] partial JSON unreadable ({_e}) -> starting fresh')
        results = {}
else:
    results = {}

def _is_config_complete(name: str) -> bool:
    r = results.get(name)
    if not r:
        return False
    return len(r.get('pearson_trajectory', [])) >= SMOKE_EPOCHS_PER_CONFIG

def _dump_partial():
    try:
        SMOKE_OUT_DIR.mkdir(parents=True, exist_ok=True)
        PARTIAL_PATH.write_text(_json.dumps(results, indent=2, default=str))
    except Exception as _e:
        print(f'  [warn] partial dump failed: {_e}')


def _use_amp_from_mode(mode: str) -> bool:
    return mode != 'fp32'


for cfg in CONFIGS:
    name = cfg['name']

    if _is_config_complete(name):
        r = results[name]
        print()
        print(f'[Cell 7] SKIP [{name}] : already complete '
              f'(ep3 pearson={r["pearson_trajectory"][-1]:.4f})')
        continue

    print()
    print('=' * 78)
    print(f'SMOKE [{name}] {cfg["description"]}')
    print(f'  warm_start={cfg["warm_start"]}  lr={cfg["lr"]}  '
          f'warmup={cfg["warmup_steps"]}  bs={cfg["batch_size"]}  '
          f'amp={cfg["amp_mode"]}  grad_ckpt={cfg["gradient_checkpointing"]}')
    print('=' * 78)

    # Fresh Stage 2 decoder.
    diff_decoder = build_fresh_stage2(
        warm_start=cfg['warm_start'],
        gradient_checkpointing=cfg['gradient_checkpointing'],
    )

    # Optimizer.
    optimizer = torch.optim.AdamW(
        diff_decoder.parameters(),
        lr=cfg['lr'],
        betas=(0.9, 0.999),
        weight_decay=cfg['weight_decay'],
    )

    # Optional warmup scheduler.
    scheduler = None
    if cfg['warmup_steps'] > 0:
        from torch.optim.lr_scheduler import LinearLR
        scheduler = LinearLR(
            optimizer,
            start_factor=0.01,
            end_factor=1.0,
            total_iters=int(cfg['warmup_steps']),
        )

    # We do NOT mid-config resume (decoder state not checkpointed in smoke).
    # If the partial JSON has 1-2 epochs for this config, we discard them
    # and restart from epoch 1 -- mixing trajectories from different decoders
    # would invalidate the comparison.
    _prev_ep = len(results.get(name, {}).get('pearson_trajectory', []))
    if 0 < _prev_ep < SMOKE_EPOCHS_PER_CONFIG:
        print(f'  [Cell 7] discarding partial ({_prev_ep} ep) for [{name}] -- restarting fresh.')
    pearson_traj, rmse_traj, loss_traj, epoch_times = [], [], [], []

    for ep in range(1, SMOKE_EPOCHS_PER_CONFIG + 1):
        _t0 = time.time()
        metrics = train_epoch_stage2_cached(
            diffusion_decoder=diff_decoder,
            optimizer=optimizer,
            cached_dataloader=train_cached_dataloader,
            device=DEVICE,
            use_amp=_use_amp_from_mode(cfg['amp_mode']),
            gradient_clipping=1.0,
            log_every=20,
            verbose=True,
            lambda_contrastive_dag=0.0,
            ema_model=None,
            conditioning_dropout_prob=0.0,
            log_loss_components=True,
        )
        if scheduler is not None:
            scheduler.step()

        ep_time = time.time() - _t0
        epoch_times.append(ep_time)
        loss_traj.append(float(metrics.get('loss_diff', float('nan'))))

        # Light eval.
        _t1 = time.time()
        eval_metrics = light_eval(diff_decoder, val_cached_dataloader)
        eval_time = time.time() - _t1
        pearson_traj.append(float(eval_metrics['pearson']))
        rmse_traj.append(float(eval_metrics['rmse']))

        print(f'  [ep{ep}/{SMOKE_EPOCHS_PER_CONFIG}] '
              f'loss={loss_traj[-1]:.4f}  '
              f'pearson={pearson_traj[-1]:.4f}  '
              f'rmse={rmse_traj[-1]:.4f}  '
              f'train_t={ep_time:.0f}s  eval_t={eval_time:.0f}s')

        # ---- PERSIST after each epoch ----
        results[name] = {
            'config': {k: v for k, v in cfg.items()},
            'pearson_trajectory': pearson_traj,
            'rmse_trajectory':    rmse_traj,
            'loss_trajectory':    loss_traj,
            'epoch_times':        epoch_times,
            'final_pearson': pearson_traj[-1] if pearson_traj else float('nan'),
            'final_rmse':    rmse_traj[-1]    if rmse_traj    else float('nan'),
        }
        _dump_partial()

    # Free memory before next config.
    del diff_decoder, optimizer
    if scheduler is not None:
        del scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print(f'[Cell 7] All configs handled. Partial state : {PARTIAL_PATH}')


In [ ]:
# === Cell 8 : Comparison table + GO/NO-GO recommendation ===
import json
from pathlib import Path as _Path_c8

# Bootstrap results from partial JSON if not in memory (fresh session).
try:
    results
except NameError:
    _partial = _Path_c8(str(SMOKE_OUT_DIR)) / 'smoke_partial.json'
    if _partial.exists():
        results = json.loads(_partial.read_text())
        print(f'[Cell 8] Loaded results from {_partial} -> {list(results.keys())}')
    else:
        raise RuntimeError(f'No results in memory and no {_partial} -- run Cell 7 first.')

print()
print('=' * 92)
print('PHASE 7 SMOKE COMPARE - RESULTS')
print('=' * 92)
header = (f'{"Config":<25}{"ep1 pearson":>12}{"ep2 pearson":>12}'
          f'{"ep3 pearson":>12}{"ep3 rmse":>10}{"avg t/ep":>10}')
print(header)
print('-' * 92)
for name, r in results.items():
    pj = r['pearson_trajectory']
    avg_time = sum(r['epoch_times']) / max(len(r['epoch_times']), 1)
    p1 = pj[0] if len(pj) > 0 else float('nan')
    p2 = pj[1] if len(pj) > 1 else float('nan')
    p3 = pj[2] if len(pj) > 2 else float('nan')
    print(f'{name:<25}{p1:>12.4f}{p2:>12.4f}{p3:>12.4f}'
          f'{r["final_rmse"]:>10.4f}{avg_time:>9.1f}s')

# ---- Decision logic ----
print()
print('DECISION CRITERIA (target = beat non-causal : Pearson >= '
      f'{NONCAUSAL_TARGET_PEARSON})')
print('-' * 92)

best_name = None
best_score = -1e18
for name, r in results.items():
    pj = r['pearson_trajectory']
    cfg = r['config']
    if not pj:
        verdict = 'NO DATA'
        projected = float('nan')
    else:
        p3 = pj[-1]
        slope = ((pj[-1] - pj[0]) / max(len(pj) - 1, 1)) if len(pj) >= 2 else 0.0

        if cfg['warm_start']:
            # Warm-start path: should reach 0.70+ at ep3.
            if p3 >= 0.75:
                verdict = 'STRONG  -> beat non-causal with 30-50 ep'
            elif p3 >= 0.65:
                verdict = 'OK      -> may beat non-causal with 100 ep'
            else:
                verdict = 'WEAK    -> warm-start stuck, prefer from-scratch'
            projected = p3 + slope * (50 - SMOKE_EPOCHS_PER_CONFIG)
        else:
            # From-scratch: extrapolate to 200 epochs.
            projected = p3 + slope * (200 - SMOKE_EPOCHS_PER_CONFIG)
            if projected >= NONCAUSAL_TARGET_PEARSON:
                verdict = (f'STRONG  -> projected 200ep pearson={projected:.3f} '
                           f'> {NONCAUSAL_TARGET_PEARSON}')
            elif projected >= 0.75:
                verdict = f'OK      -> projected 200ep pearson={projected:.3f}'
            else:
                verdict = (f'WEAK    -> projected 200ep pearson={projected:.3f} '
                           f'< {NONCAUSAL_TARGET_PEARSON}')

    # Score: projected pearson, penalised by epoch time (lightly).
    score = projected if projected == projected else -1e9  # NaN-safe
    if score > best_score:
        best_score = score
        best_name = name

    print(f'{name:<25}{verdict}')
    r['projected_pearson'] = float(projected) if projected == projected else None
    r['verdict'] = verdict

print()
print('=' * 92)
if best_name is not None:
    print(f'RECOMMENDED CONFIG : {best_name}  (projected pearson={best_score:.4f})')
else:
    print('RECOMMENDED CONFIG : -- (no valid results)')
print('=' * 92)

# Save results to JSON.
SMOKE_OUT_DIR.mkdir(parents=True, exist_ok=True)
_out_path = SMOKE_OUT_DIR / 'smoke_comparison.json'
_serialisable = {
    'results': results,
    'noncausal_target_pearson': NONCAUSAL_TARGET_PEARSON,
    'smoke_epochs_per_config': SMOKE_EPOCHS_PER_CONFIG,
    'light_eval': {
        'n_batches': LIGHT_EVAL_N_BATCHES,
        'k_samples': LIGHT_EVAL_K_SAMPLES,
        'n_steps':   LIGHT_EVAL_N_STEPS,
    },
    'recommended_config': best_name,
    'recommended_projected_pearson': float(best_score) if best_name else None,
}
_out_path.write_text(json.dumps(_serialisable, indent=2, default=str))
print()
print(f'Results saved : {_out_path}')
print()
print('NEXT STEP : take the recommended config, then build')
print('  phase7_stage2_retrain_final.ipynb with that config for 200 epochs.')


In [ ]:
# === Cell 9 : Compute / cost summary + sanity recap ===
print('=' * 70)
print('PHASE 7 SMOKE COMPARE - COMPUTE ESTIMATE')
print('=' * 70)
print()
print('Per-config budget (A100, batch_size=64, no gradient_checkpointing) :')
print('  - Stage 2 train epoch (cached, ~6230 samples / 97 batches) : ~3 min')
print('  - Light eval (4 batches x 16 K-samples x 18 steps)         : ~1.5 min')
print('  - Per epoch total                                          : ~4.5 min')
print('  - 3 epochs / config                                        : ~14 min')
print()
print('Total smoke (3 configs)                                      : ~42 min')
print('mu_total precompute (one-time, ~6927 train samples)          : ~30-45 min')
print()
print('=> FIRST RUN  (cold cache)  : ~75-90 min on A100')
print('=> WARM RUNS  (cache exists): ~45 min on A100')
print()
print('Memory footprint :')
print('  - Cache file on Drive   : ~1.4 GB (4 tensors x 6927 x 1 x 172 x 179 x fp32)')
print('  - GPU peak (BS=64, no_ckpt, bf16) : ~35-40 GB on A100 80GB')
print()
print('Caveats / decisions baked in :')
print('  - tail_weight (tau95=15mm/8x, tau99=35mm/25x) inherited from')
print('    CONFIG.diffusion.edm (corrdiff_normal yaml). No override needed.')
print('  - S_churn=40 (non-causal sampler choice) inherited the same way.')
print('  - sigma_data=0.193 overridden post-build (Phase 6 dualpath value).')
print('  - EMA OFF (smoke; full-scale run will turn it on).')
print('  - conditioning_dropout=0 (smoke).')
print('  - lambda_contrastive_dag=0 (Stage 1 frozen, no DAG variants in cache).')
print('  - train_epoch_stage2_cached uses log_every (not log_interval), and')
print('    returns "loss_diff" (not "loss").')
print('  - use_amp=True -> resolve_train_amp_mode picks cuda_bf16 on A100.')
print()
print('Output artefacts :')
print(f'  - {STAGE1_CACHE_PATH}')
print(f'  - {SMOKE_OUT_DIR / "smoke_comparison.json"}')


In [ ]:
# === Cell 10 (OPTIONAL DIAGNOSTIC) : run only if smoke STILL stalls ===
#
# Replays first 80 batches manually with per-batch CUDA-synced timing.
# Self-contained: builds a fresh Stage 2 + uses local cache + manual indexing.
#
# READING THE OUTPUT:
# - Uniform per-batch times (~0.5-1.5s) up to batch 80 -> stall is OUTSIDE
#     the train loop (cache reload, gc, validation, EMA, scheduler)
# - One specific batch is 10x slower -> data-driven. Check that batch stats.
# - Same SAMPLE INDEX slow in shuffle=False -> corrupted sample
# - bf16 hangs but fp32 OK -> bf16 numerical issue
# - GPU memory grows monotonically -> leak

import time as _t, gc, math
import torch
from pathlib import Path
from omegaconf import OmegaConf as _OC
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig

assert torch.cuda.is_available(), "Need CUDA"
torch.cuda.empty_cache(); gc.collect()
print(f"[diag] CUDA mem at entry : {torch.cuda.memory_allocated()/1e9:.2f} GB alloc, "
      f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

# [0] Reload cache from local SSD (assumes Cell 5 already ran once)
CACHE_LOCAL = Path("/content/stage1_cache_phase7_dualpath.pt")
assert CACHE_LOCAL.exists(), f"Local cache missing: {CACHE_LOCAL}. Run Cell 5 first."
cache_diag = torch.load(CACHE_LOCAL, map_location="cpu", weights_only=False)
cache_diag = {k: v.contiguous().clone() for k, v in cache_diag.items()}
N = cache_diag["mu_HR"].shape[0]
print(f"[diag] Cache loaded : N={N}")

# [1] Per-sample stats
print()
print("=" * 78)
print("[1] PER-SAMPLE STATS (top outliers + NaN/Inf flags)")
print("=" * 78)
mu, base, delt = cache_diag["mu_HR"], cache_diag["baseline_log"], cache_diag["delta_target"]
flat_mu = mu.view(N, -1).float()
flat_dlt = delt.view(N, -1).float()
mu_max = flat_mu.amax(dim=1)
dlt_absmax = flat_dlt.abs().amax(dim=1)
nan_mask = (~torch.isfinite(flat_mu)).any(dim=1) | (~torch.isfinite(flat_dlt)).any(dim=1)
print(f"  global mu_HR        : min={mu.min():.4f}  max={mu.max():.4f}")
print(f"  global delta_target : min={delt.min():.4f}  max={delt.max():.4f}")
print(f"  samples with NaN/Inf : {int(nan_mask.sum())}")
if nan_mask.any():
    print(f"    indices: {torch.nonzero(nan_mask).flatten().tolist()[:20]}")
top_dlt = torch.topk(dlt_absmax, 10).indices.tolist()
print(f"  top 10 |delta| indices : {top_dlt}")

# [2] Fresh Stage 2 decoder
print()
print("=" * 78)
print("[2] BUILDING FRESH Stage 2 decoder")
print("=" * 78)
hr_channels = int(cache_diag["delta_target"].shape[1])
edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {}))
edm_cfg.sigma_data = 0.193
UNET_KW = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ("down_block_types", "up_block_types"):
    if _k in UNET_KW and isinstance(UNET_KW[_k], list):
        UNET_KW[_k] = tuple(UNET_KW[_k])
UNET_KW["projection_class_embeddings_input_dim"] = num_vars * int(CONFIG.diffusion.conditioning_dim)
dd = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=int(CONFIG.diffusion.height), width=int(CONFIG.diffusion.width),
    unet_kwargs=UNET_KW,
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    use_gradient_checkpointing=True,
    conv_padding_mode=str(CONFIG.diffusion.get("conv_padding_mode", "zeros")),
    anti_checkerboard=bool(CONFIG.diffusion.get("anti_checkerboard", False)),
    edm_config=edm_cfg, causal_concat=True,
).to(DEVICE)
dd.train()
opt = torch.optim.AdamW(dd.parameters(), lr=2e-4)
print(f"  decoder params : {sum(p.numel() for p in dd.parameters()):,}")

# [3] Helper to build batch + time one step
def _build_batch(idxs):
    return {
        "mu_HR":        cache_diag["mu_HR"][idxs].to(DEVICE, non_blocking=True),
        "baseline_log": cache_diag["baseline_log"][idxs].to(DEVICE, non_blocking=True),
        "delta_target": cache_diag["delta_target"][idxs].to(DEVICE, non_blocking=True),
    }

def _time_step(batch, use_bf16=True, sigma_override=None):
    opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize(); t0 = _t.perf_counter()
    ctx = (torch.autocast("cuda", dtype=torch.bfloat16) if use_bf16
           else torch.autocast("cuda", enabled=False))
    with ctx:
        from st_cdgm.models.edm_preconditioner import sample_training_sigma
        B = batch["delta_target"].shape[0]
        if sigma_override is None:
            sigma = sample_training_sigma(B,
                P_mean=edm_cfg.P_mean, P_std=edm_cfg.P_std,
                device=DEVICE, dtype=batch["delta_target"].dtype)
        else:
            sigma = torch.full((B,), float(sigma_override),
                               device=DEVICE, dtype=batch["delta_target"].dtype)
        noise = torch.randn_like(batch["delta_target"])
        y_noisy = batch["delta_target"] + sigma.view(-1,1,1,1) * noise
        D_y = dd.forward_edm(y_noisy, sigma, conditioning=None,
            conditioning_spatial=None,
            mu_HR=batch["mu_HR"], baseline_log=batch["baseline_log"])
        w = (sigma**2 + edm_cfg.sigma_data**2) / (sigma * edm_cfg.sigma_data)**2
        loss = (w.view(-1,1,1,1) * (D_y - batch["delta_target"])**2).mean()
    torch.cuda.synchronize(); t_fwd = _t.perf_counter() - t0
    t0 = _t.perf_counter()
    loss.backward()
    torch.cuda.synchronize(); t_bwd = _t.perf_counter() - t0
    return float(loss.detach()), sigma, t_fwd, t_bwd

# [4] Replay 80 batches with shuffle (seed=42)
print()
print("=" * 78)
print("[4] REPLAYING 80 batches  shuffle=True  seed=42  bf16")
print("=" * 78)
g = torch.Generator().manual_seed(42)
perm = torch.randperm(N, generator=g).tolist()
BS = 64
n_run = min(80, len(perm) // BS)
mem_log = []
for b in range(n_run):
    idxs = perm[b*BS:(b+1)*BS]
    batch = _build_batch(idxs)
    loss, sigma, t_fwd, t_bwd = _time_step(batch, use_bf16=True)
    opt.step()
    mem = torch.cuda.memory_allocated() / 1e9
    mem_log.append(mem)
    is_slow = (t_fwd + t_bwd) > 3.0
    if b < 5 or b >= n_run - 5 or b % 10 == 0 or is_slow:
        suffix = "  <<< SLOW" if is_slow else ""
        print(f"  batch {b:3d} | t_fwd={t_fwd*1000:6.0f}ms t_bwd={t_bwd*1000:6.0f}ms "
              f"| loss={loss:7.4f} | sigma_max={sigma.max():.2f} "
              f"| mem={mem:5.2f}GB{suffix}")

if len(mem_log) >= 2:
    growth = mem_log[-1] - mem_log[0]
    leak_msg = "LEAK SUSPECT" if growth > 1.0 else "stable"
    print(f"\n  mem growth over {len(mem_log)} batches : {growth:+.3f} GB  ({leak_msg})")

# [5] Shuffle=False (test sample-index vs batch-index)
print()
print("=" * 78)
print("[5] REPLAY shuffle=False (20 batches)")
print("=" * 78)
for b in range(min(20, N // BS)):
    idxs = list(range(b*BS, (b+1)*BS))
    batch = _build_batch(idxs)
    loss, sigma, t_fwd, t_bwd = _time_step(batch, use_bf16=True)
    opt.step()
    if b < 3 or b >= 17 or (t_fwd+t_bwd) > 3.0:
        print(f"  batch {b:3d} (samples {idxs[0]}-{idxs[-1]}) | "
              f"t={1000*(t_fwd+t_bwd):6.0f}ms loss={loss:.4f} sigma_max={sigma.max():.2f}")

# [6] FP32 fallback
print()
print("=" * 78)
print("[6] FP32 FALLBACK probe (batch 0)")
print("=" * 78)
idxs = perm[:BS]
batch = _build_batch(idxs)
loss, sigma, t_fwd, t_bwd = _time_step(batch, use_bf16=False)
print(f"  fp32 batch 0 | t={1000*(t_fwd+t_bwd):.0f}ms loss={loss:.4f}")

# [7] Extreme sigma probe
print()
print("=" * 78)
print("[7] EXTREME-SIGMA probe")
print("=" * 78)
loss, sigma, t_fwd, t_bwd = _time_step(batch, use_bf16=True, sigma_override=80.0)
print(f"  sigma=80 forced | t={1000*(t_fwd+t_bwd):.0f}ms loss={loss:.4f}")
loss, sigma, t_fwd, t_bwd = _time_step(batch, use_bf16=True, sigma_override=0.002)
print(f"  sigma=0.002 forced | t={1000*(t_fwd+t_bwd):.0f}ms loss={loss:.4f}")

print()
print("[diag] Done. See header for output interpretation.")
print("[diag] Free memory : del dd, opt; gc.collect(); torch.cuda.empty_cache()")


In [ ]:
# === Cell 11 (DISCRIMINANT TEST) : DataLoader vs compute_loss_edm ===
#
# Cell 10 proved manual indexing + forward_edm INLINE = no stall.
# Cell 7 (DataLoader + compute_loss_edm) = stall at batch 60.
# Two suspects left. This cell runs them in ISOLATION over 100+ batches
# (well past batch 60).
#
# Test A : DataLoader iter  + INLINE forward+EDM loss   -> isolates DataLoader
# Test B : MANUAL perm/idx  + compute_loss_edm(comp=T)  -> isolates compute_loss_edm
#
# To pass batch 60 cleanly we cycle the DataLoader (Test A) and use the
# full cache N=5468 (Test B), giving ~150 and ~85 batches respectively.

import time as _t_disc, gc as _gc_disc
import torch as _torch_disc
from st_cdgm.models.edm_preconditioner import sample_training_sigma as _samp_sig

assert 'train_cached_dataloader' in globals(), "Run Cell 5 first."
assert 'cache' in globals(), "Run Cell 5 first."

_TARGET_BATCHES = 100
_BS_DISC = 64

# ---------------------------------------------------------------------------
# Test A : DataLoader iterator + inline forward+EDM loss
# ---------------------------------------------------------------------------
print("=" * 78)
print(f"[TEST A] DataLoader + INLINE forward+EDM loss  (target {_TARGET_BATCHES} batches)")
print("=" * 78)

dd_A = build_fresh_stage2(warm_start=False, gradient_checkpointing=True)
opt_A = _torch_disc.optim.AdamW(dd_A.parameters(), lr=2e-4)
_edm_A = dd_A.edm_config

_times_A = []
_b_done_A = 0
_pass = 0
while _b_done_A < _TARGET_BATCHES:
    _pass += 1
    for _b_iter, batch in enumerate(train_cached_dataloader):
        b = _b_done_A
        if b >= _TARGET_BATCHES:
            break
        mu_HR = batch['mu_HR'].to(DEVICE, non_blocking=True)
        base  = batch['baseline_log'].to(DEVICE, non_blocking=True)
        dlt   = batch['delta_target'].to(DEVICE, non_blocking=True)
        opt_A.zero_grad(set_to_none=True)
        _torch_disc.cuda.synchronize(); _t0 = _t_disc.perf_counter()
        with _torch_disc.autocast('cuda', dtype=_torch_disc.bfloat16):
            B = dlt.shape[0]
            sig = _samp_sig(B, P_mean=_edm_A.P_mean, P_std=_edm_A.P_std,
                            device=DEVICE, dtype=dlt.dtype)
            noise = _torch_disc.randn_like(dlt)
            y = dlt + sig.view(-1, 1, 1, 1) * noise
            D_y = dd_A.forward_edm(y, sig, conditioning=None, conditioning_spatial=None,
                                   mu_HR=mu_HR, baseline_log=base)
            w = (sig**2 + _edm_A.sigma_data**2) / (sig * _edm_A.sigma_data)**2
            _loss_val = (w.view(-1, 1, 1, 1) * (D_y - dlt)**2).mean()
        _loss_val.backward()
        opt_A.step()
        _torch_disc.cuda.synchronize(); _dt = _t_disc.perf_counter() - _t0
        _times_A.append(_dt)
        if b < 3 or b % 10 == 0 or 55 <= b <= 75 or _dt > 3.0:
            _slow = "  <<< SLOW" if _dt > 3.0 else ""
            print(f"  A batch {b:3d} (pass {_pass}) | t={_dt*1000:6.0f}ms "
                  f"loss={float(_loss_val.detach()):.4f}{_slow}", flush=True)
        _b_done_A += 1

print(f"[TEST A] DONE : {_b_done_A} batches | max={max(_times_A)*1000:.0f}ms "
      f"| mean={sum(_times_A)/len(_times_A)*1000:.0f}ms")

del dd_A, opt_A
_gc_disc.collect()
_torch_disc.cuda.empty_cache()

# ---------------------------------------------------------------------------
# Test B : manual perm + compute_loss_edm (with components)
# ---------------------------------------------------------------------------
print()
print("=" * 78)
print(f"[TEST B] MANUAL perm + compute_loss_edm(return_components=True)  (target {_TARGET_BATCHES} batches)")
print("=" * 78)

dd_B = build_fresh_stage2(warm_start=False, gradient_checkpointing=True)
opt_B = _torch_disc.optim.AdamW(dd_B.parameters(), lr=2e-4)

_N = int(cache['mu_HR'].shape[0])
_g_B = _torch_disc.Generator().manual_seed(123)
_perm_B = _torch_disc.randperm(_N, generator=_g_B).tolist()
# Cycle the perm so we exceed N/BS when needed.
_max_b = _TARGET_BATCHES
_full_cycle = _N // _BS_DISC

_times_B = []
for b in range(_max_b):
    _i_cycle = b % _full_cycle
    _idxs = _perm_B[_i_cycle*_BS_DISC:(_i_cycle+1)*_BS_DISC]
    mu_HR = cache['mu_HR'][_idxs].to(DEVICE, non_blocking=True)
    base  = cache['baseline_log'][_idxs].to(DEVICE, non_blocking=True)
    dlt   = cache['delta_target'][_idxs].to(DEVICE, non_blocking=True)
    opt_B.zero_grad(set_to_none=True)
    _torch_disc.cuda.synchronize(); _t0 = _t_disc.perf_counter()
    with _torch_disc.autocast('cuda', dtype=_torch_disc.bfloat16):
        _out = dd_B.compute_loss_edm(
            target=dlt, conditioning=None, conditioning_spatial=None,
            mu_HR=mu_HR, baseline_log=base, return_components=True,
        )
        _loss_val, _comps = _out
    _loss_val.backward()
    opt_B.step()
    _torch_disc.cuda.synchronize(); _dt = _t_disc.perf_counter() - _t0
    _times_B.append(_dt)
    if b < 3 or b % 10 == 0 or 55 <= b <= 75 or _dt > 3.0:
        _slow = "  <<< SLOW" if _dt > 3.0 else ""
        print(f"  B batch {b:3d} | t={_dt*1000:6.0f}ms "
              f"loss={float(_loss_val.detach()):.4f}{_slow}", flush=True)

print(f"[TEST B] DONE : {_max_b} batches | max={max(_times_B)*1000:.0f}ms "
      f"| mean={sum(_times_B)/len(_times_B)*1000:.0f}ms")

del dd_B, opt_B
_gc_disc.collect()
_torch_disc.cuda.empty_cache()

print()
print("=" * 78)
print("INTERPRETATION")
print("=" * 78)
print("  A stalled,  B OK    -> DataLoader is the culprit")
print("  B stalled,  A OK    -> compute_loss_edm (or components) is the culprit")
print("  BOTH stalled        -> combination / environment (cuDNN, driver, autograd cache)")
print("  NEITHER stalled     -> stall is in Cell 7 boilerplate around the loop")


In [ ]:
# === Cell 12 (DISCRIMINANT TEST C) : direct train_epoch_stage2_cached x 2 epochs ===
#
# Cell 11 ruled out DataLoader-alone and compute_loss_edm-alone (each 100+ batches OK).
# Test C calls train_epoch_stage2_cached() directly with the EXACT args Cell 7 uses.
# We run 2 epochs back-to-back to exceed ~150 batches and be confident past batch 60.
#
# Outcomes :
#   C stalls at batch ~60 (1st or 2nd epoch) -> the function itself is the culprit
#   C completes BOTH epochs -> some surrounding context in Cell 7 is the culprit

import time as _t_C, gc as _gc_C
import torch as _torch_C
from st_cdgm.training.two_stage import train_epoch_stage2_cached

assert 'train_cached_dataloader' in globals(), "Run Cell 5 first."

print("=" * 78)
print("[TEST C] train_epoch_stage2_cached() direct call -- 2 epochs (mirrors Cell 7 args)")
print("=" * 78)

dd_C = build_fresh_stage2(warm_start=False, gradient_checkpointing=True)
opt_C = _torch_C.optim.AdamW(
    dd_C.parameters(),
    lr=2e-4,
    betas=(0.9, 0.999),
    weight_decay=1e-4,
)

_metrics_per_ep = []
for _ep in range(1, 3):
    print()
    print(f"--- Epoch {_ep}/2 ---")
    _t0 = _t_C.perf_counter()
    metrics = train_epoch_stage2_cached(
        diffusion_decoder=dd_C,
        optimizer=opt_C,
        cached_dataloader=train_cached_dataloader,
        device=DEVICE,
        use_amp=True,
        gradient_clipping=1.0,
        log_every=20,
        verbose=True,
        lambda_contrastive_dag=0.0,
        ema_model=None,
        conditioning_dropout_prob=0.0,
        log_loss_components=True,
    )
    _dt = _t_C.perf_counter() - _t0
    _metrics_per_ep.append((metrics, _dt))
    print(f"[ep {_ep}] DONE in {_dt:.1f}s | loss_diff={metrics.get('loss_diff', float('nan')):.4f}")

print()
print("[TEST C] BOTH EPOCHS COMPLETED -> train_epoch_stage2_cached is NOT the culprit.")

del dd_C, opt_C
_gc_C.collect()
_torch_C.cuda.empty_cache()

print()
print("=" * 78)
print("INTERPRETATION")
print("=" * 78)
print("  C stalled  -> train_epoch_stage2_cached boilerplate is the culprit")
print("                (scaler obj / _train_autocast / .item() chain / clip_grad_norm)")
print("  C completed both epochs -> stall must be OUTSIDE the function.")
print("                             Compare what Cell 7 does between configs that we don't here :")
print("                             - it calls light_eval() between train+next epoch")
print("                             - it runs 3 configs x 3 epochs with del+gc.collect+empty_cache between")
print("                             - it has the LinearLR scheduler step in config C")


In [ ]:
# === Cell 13 (DISCRIMINANT TEST D) : train_epoch + light_eval ===
#
# Test C (train_epoch_stage2_cached x 2 epochs) did NOT stall.
# Cell 7 differs from Test C by calling light_eval() BETWEEN epochs.
#
# Cell 7 trace shows last log "batch 60" then stall for 5+ min.
# Epoch has 77 batches (4921/64); log_every=20 so batches 61-76 are
# silent before the epoch ends. light_eval runs immediately after.
#
# This cell instruments light_eval with per-K-sample timing to see
# exactly where (if at all) it stalls.

import time as _t_D, gc as _gc_D
import torch as _torch_D
from st_cdgm.training.two_stage import train_epoch_stage2_cached

assert 'train_cached_dataloader' in globals(), "Run Cell 5 first."
assert 'val_cached_dataloader'   in globals(), "Run Cell 5 first."

print("=" * 78)
print("[TEST D] train_epoch + light_eval -- traces every eval forward pass")
print("=" * 78)

dd_D = build_fresh_stage2(warm_start=False, gradient_checkpointing=True)
opt_D = _torch_D.optim.AdamW(dd_D.parameters(), lr=2e-4, betas=(0.9, 0.999), weight_decay=1e-4)

# --- 1 train epoch (~100s on A100) ---
print()
print("--- Phase 1/2 : train_epoch_stage2_cached x 1 epoch ---")
_t0 = _t_D.perf_counter()
metrics_D = train_epoch_stage2_cached(
    diffusion_decoder=dd_D, optimizer=opt_D,
    cached_dataloader=train_cached_dataloader, device=DEVICE,
    use_amp=True, gradient_clipping=1.0, log_every=20, verbose=True,
    lambda_contrastive_dag=0.0, ema_model=None,
    conditioning_dropout_prob=0.0, log_loss_components=True,
)
_dt_train = _t_D.perf_counter() - _t0
print(f"[TEST D] train_epoch done in {_dt_train:.1f}s | loss_diff={metrics_D.get('loss_diff', float('nan')):.4f}")

# --- INSTRUMENTED light_eval (mirrors Cell 6 but prints per K and per batch) ---
print()
print(f"--- Phase 2/2 : light_eval -- {LIGHT_EVAL_N_BATCHES} batches x "
      f"{LIGHT_EVAL_K_SAMPLES} K_samples x {LIGHT_EVAL_N_STEPS} steps ---")
_t_eval0 = _t_D.perf_counter()
dd_D.eval()

preds, targets = [], []
n_done = 0

for _b_eval, batch in enumerate(val_cached_dataloader):
    if n_done >= LIGHT_EVAL_N_BATCHES:
        break
    print(f"  [eval batch {_b_eval}] starting (n_done={n_done}/{LIGHT_EVAL_N_BATCHES})", flush=True)
    mu_HR_e        = batch['mu_HR'].to(DEVICE)
    baseline_log_e = batch['baseline_log'].to(DEVICE)
    delta_target_e = batch['delta_target'].to(DEVICE)

    samples_e = []
    _t_b0 = _t_D.perf_counter()
    for _k in range(LIGHT_EVAL_K_SAMPLES):
        _torch_D.cuda.synchronize(); _tk0 = _t_D.perf_counter()
        with _torch_D.no_grad():
            out = dd_D.sample(
                conditioning=None,
                num_steps=LIGHT_EVAL_N_STEPS,
                scheduler_type='edm_karras',
                apply_constraints=False,
                mu_HR=mu_HR_e,
                baseline_log=baseline_log_e,
            )
        _torch_D.cuda.synchronize(); _tk = _t_D.perf_counter() - _tk0
        res_e = out.residual if hasattr(out, 'residual') else out
        samples_e.append(res_e)
        print(f"    K={_k:2d}/{LIGHT_EVAL_K_SAMPLES} | t={_tk*1000:6.0f}ms", flush=True)

    ens_e = _torch_D.stack(samples_e, dim=0)
    delta_pred_e = ens_e.mean(dim=0)
    pred_full = baseline_log_e + mu_HR_e + delta_pred_e
    target_full = baseline_log_e + mu_HR_e + delta_target_e
    preds.append(pred_full.cpu())
    targets.append(target_full.cpu())
    n_done += 1
    _t_b = _t_D.perf_counter() - _t_b0
    print(f"  [eval batch {_b_eval}] done in {_t_b:.1f}s", flush=True)

dd_D.train()

p = _torch_D.cat(preds,   dim=0).flatten()
t = _torch_D.cat(targets, dim=0).flatten()
valid = _torch_D.isfinite(t) & _torch_D.isfinite(p)
pv = p[valid]; tv = t[valid]
pm, tm = pv.mean(), tv.mean()
num = ((pv - pm) * (tv - tm)).sum()
den = (((pv - pm) ** 2).sum() * ((tv - tm) ** 2).sum()).sqrt()
pearson = float(num / max(den, _torch_D.tensor(1e-12)))
rmse    = float((pv - tv).pow(2).mean().sqrt())

_dt_eval = _t_D.perf_counter() - _t_eval0
print()
print(f"[TEST D] light_eval done in {_dt_eval:.1f}s | pearson={pearson:.4f} | rmse={rmse:.4f}")

del dd_D, opt_D
_gc_D.collect()
_torch_D.cuda.empty_cache()

print()
print("=" * 78)
print("INTERPRETATION")
print("=" * 78)
print("  Stall during light_eval -> EDM Karras sampler is the culprit (Heun ODE)")
print("    -- look at which K sample / batch stalled and its timing pattern")
print("  Completes normally -> light_eval is just SLOW (not stuck), Cell 7 is fine")
print("    -- reduce LIGHT_EVAL_K_SAMPLES (16->4) or LIGHT_EVAL_N_BATCHES (4->2) to speed up")
